# Phase 5 step 2 — Mask2Former + finer Mask R-CNN mask head (A100)

Two new models, each a new member for the ensemble (current best: x101 + r50 fused, val PQ 0.4366):

| Step | Name | What | Time on A100 (estimate) |
|---|---|---|---|
| 0 | probes | 60 Mask2Former steps + 20 Mask R-CNN steps; prints GPU memory and speed | ~15 min |
| 1 | `m2f_swin_s` | **Mask2Former** (Swin-S, COCO weights): no boxes, 100 queries each paint one filament; trained on 1024 crops of the native 2048 images, predicts on the full image | ≤ 4 h (capped automatically) |
| 2 | `x101_2048_m28` | `x101_2048` recipe, mask grid 28×28 → **56×56** per box (sharper thin filaments) | ~3 h |

**Runtime:** A100 GPU + High-RAM. ~8 compute units per hour, ~55–60 units in total. Close other Colab sessions.
Keep this tab open.

**How to run:** Runtime → Run all. If the Mask2Former probe fails, step 1 is skipped and step 2 still runs (send
Claude the error). Raw predictions are copied to `MyDrive/solar_filament/Phase 5/` as soon as each step finishes, so
the step 1 files can be sent back while step 2 trains.

**If Colab disconnects:** run sections 1–4 again, then re-run the cell of the interrupted step: both trainings
resume from their last checkpoint on Drive (every 1,000 iterations).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
OUT_DIR = f"{DRIVE_DIR}/output"            # checkpoints (persistent)
PHASE5_DIR = f"{DRIVE_DIR}/Phase 5"         # raw predictions to send back
os.makedirs(PHASE5_DIR, exist_ok=True)
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Drive ready:", OUT_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5-8 min)
`transformers` is pinned to the version the Mask2Former code was tested with.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q transformers==5.17.0
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2, transformers
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| transformers", transformers.__version__,
      "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~1 min) and helpers

In [ ]:
import re, shutil, subprocess, sys, time

def sh(cmd, capture=False):
    """Run one command (output shown live); stop the notebook if it fails. capture=True also returns the output."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")
    return "".join(lines) if capture else None

def save_raw(name):
    for split in ("val", "test"):
        src = f"{OUT_DIR}/raw_preds_{name}_{split}.json.gz"
        shutil.copy(src, PHASE5_DIR)
        print(split, os.path.getsize(src) // 1024, "KB ->", PHASE5_DIR)

def remove_probe(name, subdir):
    shutil.rmtree(f"{OUT_DIR}/{subdir}/{name}", ignore_errors=True)
    for split in ("val", "test"):
        f = f"{OUT_DIR}/raw_preds_{name}_{split}.json.gz"
        if os.path.exists(f):
            os.remove(f)

sh("python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_official_metric.py tests/test_ensemble.py")

## Step 0 — Probes (~15 min)
**Mask2Former:** 60 training steps on 1024 crops (batch 4), then predictions on 2 images. Look for
`time ... s/it`, `Peak GPU memory` and `[Dump] ... s/img`. The training length of step 1 is set from the measured
speed so that it takes at most ~4 h. If this probe fails, step 1 is skipped (send Claude the error).

**Mask R-CNN with the finer mask head:** 20 training steps. Must pass, or the notebook stops.

In [ ]:
M2F_NAME, M2F_ITERS, M2F_HOURS = "m2f_swin_s", 12000, 4.0
M2F_OK = False
try:
    log = sh("python src/m2f.py train --name probe_m2f --max-iter 60 --no-resume", capture=True)
    sh("python src/m2f.py dump --name probe_m2f --limit 2")
    sec = float(re.findall(r"time ([0-9.]+) s/it", log)[-1])
    M2F_ITERS = min(M2F_ITERS, max(4000, int(M2F_HOURS * 3600 / sec) // 1000 * 1000))
    M2F_OK = True
    print(f"Mask2Former probe passed: {sec:.2f} s/it -> step 1 trains {M2F_ITERS} iterations "
          f"(~{M2F_ITERS * sec / 3600:.1f} h)")
except Exception as e:
    print("MASK2FORMER PROBE FAILED - step 1 will be skipped. Send Claude the error above.", e)
remove_probe("probe_m2f", "m2f")

FLAGS_M28 = "--backbone x101 --resolution 2048 --mask-res 28"
sh(f"python run_pipeline.py --step train --name probe_m28 {FLAGS_M28} --label-policy all --max-iter 20 --no-resume")
remove_probe("probe_m28", "detectron2")
print("Mask R-CNN probe passed.")

## Step 1 — Mask2Former (`m2f_swin_s`, ≤ 4 h)

In [ ]:
if M2F_OK:
    sh(f"python src/m2f.py train --name {M2F_NAME} --max-iter {M2F_ITERS}")
    sh(f"python src/m2f.py dump --name {M2F_NAME}")
    save_raw(M2F_NAME)
else:
    print("Skipped: the Mask2Former probe failed.")

## Step 2 — X101 at 2048 with the finer mask head (`x101_2048_m28`, ~3 h)

In [ ]:
NAME_M28 = "x101_2048_m28"
sh(f"python run_pipeline.py --step train --name {NAME_M28} {FLAGS_M28} --label-policy all")
sh(f"python run_pipeline.py --step dump  --name {NAME_M28} {FLAGS_M28}")
save_raw(NAME_M28)

## 5. Send results back
Download the raw prediction files (also in `MyDrive/solar_filament/Phase 5/`) into the local folder
`Result Step 5/Step 2/`, then tell Claude. Please also copy, for each step: the `Peak GPU memory` line, the last
`time ... s/it` line and the `[Dump] ... s/img` lines.
Then **disconnect the runtime** (Runtime → Disconnect and delete runtime) so it stops using compute units.

In [ ]:
from google.colab import files
for name in ("m2f_swin_s", "x101_2048_m28"):
    for split in ("val", "test"):
        f = f"{PHASE5_DIR}/raw_preds_{name}_{split}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (step not finished or skipped?):", f)